In [ ]:
import sys
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

import os

import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from IPython.display import display, Markdown, Video

from sklearn.datasets import make_moons, make_circles
from sklearn.preprocessing import StandardScaler

from scipy.stats import multivariate_normal, gaussian_kde

torch.manual_seed(1)
np.random.seed(1)

sys.path.extend(["../"])

from VAE import VAE
from utils_data import generate_banana_data


In [ ]:
display(Markdown(open("../_macros.md").read()))

# Variational Autoencoder

Need to put here the theory. But need to create first the EM theory file.

## Dataset 

Let's generate data from $p(\xvec)$ and also from the latent space $p(\zvec)$. Here we use the same dimensionality although we are not required to.

In [ ]:
# Fijar semilla para reproducibilidad
np.random.seed(42)

# Crear figura con dos subplots (1 fila, 2 columnas)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# =========================
# == Prior p(z) = N(0,I) ==
# =========================

# Generar muestras 2D de una normal estándar
mean = [0, 0]
cov = [[1, 0], [0, 1]]
samples = np.random.multivariate_normal(mean, cov, 500)

# Scatter de las muestras
axes[0].scatter(samples[:, 0], samples[:, 1], color = 'C1')

# Crear grid para curvas de nivel
x = np.linspace(-4, 4, 200)
y = np.linspace(-4, 4, 200)
X, Y = np.meshgrid(x, y)
pos = np.dstack((X, Y))

rv = multivariate_normal(mean, cov)
Z = rv.pdf(pos)

# Dibujar curvas de nivel
axes[0].contour(X, Y, Z, levels=10, cmap = 'Oranges')

axes[0].set_title(r"$p({\bf z})$")
axes[0].set_xlabel(r"x_1")
axes[0].set_ylabel(r"x_2")

# =========================
# ======== p(x) ===========
# =========================
# X_moons, y_moons = make_moons(n_samples=1000, noise=0.1, random_state=42)
# X_moons, y_moons = make_circles(n_samples=1000, noise=0.05, factor=0.5, random_state=42)
X_moons = generate_banana_data()

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], color = 'C0')
axes[1].set_title(r"$p({\bf x})$")
axes[1].set_xlabel(r"x_1")
axes[1].set_ylabel(r"x_2")

plt.tight_layout()
plt.show()


### Create Torch dataset and dataloader

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_moons)

X = torch.tensor(X_scaled, dtype=torch.float32)
dataset = TensorDataset(X)
train_loader = DataLoader(dataset, batch_size=1000, shuffle=True)
sampling_loader = DataLoader(dataset, batch_size=20, shuffle=True)

## Train a Variational Autoencoder

In [ ]:
data_dim = 2
latent_dim = 2
hidden_dim = 64

encoder_layers = [
    (data_dim, hidden_dim, nn.Tanh),
    (hidden_dim, hidden_dim, nn.Tanh),
    (hidden_dim, latent_dim, None)
]
decoder_layers = [
    (latent_dim, hidden_dim, nn.Tanh),
    (hidden_dim, hidden_dim, nn.Tanh),
    (hidden_dim, data_dim, None)
]



In [ ]:
vae = VAE(
            encoder_layers,
            decoder_layers,
            latent_dim = latent_dim,
            decoder_type = "Gaussian",
            N = X_moons.shape[0],
)

optimizer = optim.Adam(vae.parameters(), lr=1e-3)

num_epochs = 2000

for epoch in range(num_epochs):
    vae.train()
    total_elbo = 0
    total_kld = 0.0
    total_ell = 0.0
    for x_tr, in train_loader:

        optimizer.zero_grad()
        elbo, ell, kld = vae.ELBO(x_tr, mc_samples = 1, kld_scale = 1)
        loss = -elbo 
        loss.backward()
        optimizer.step()

        total_elbo += elbo.item() 
        total_ell += ell.item()
        total_kld += kld.item()

    if epoch == 200: # annealing
        for param_group in optimizer.param_groups:
            param_group['lr'] = 1e-3

    if epoch == 1200: # annealing
        for param_group in optimizer.param_groups:
            param_group['lr'] = 1e-4
        
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}, ELBO: {total_elbo:.4f}, ELL:{total_ell:.4f}, KLD: {total_kld:.4}")

### Generating Samples from the prior $p(\zvec)$

In [ ]:
# Fijar semilla para reproducibilidad
np.random.seed(42)

# Crear figura con dos filas y dos columnas
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

def grab_frame():
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

# =========================
# == Prior p(z) = N(0,I) ==
# =========================

# Generar muestras 2D de una normal estándar
mean = [0, 0]
cov = [[1, 0], [0, 1]]
samples = np.random.multivariate_normal(mean, cov, 500)

# Scatter de las muestras
axes[0, 0].scatter(samples[:, 0], samples[:, 1], color='C1')

# Crear grid para curvas de nivel
x = np.linspace(-4, 4, 200)
y = np.linspace(-4, 4, 200)
X_grid, Y_grid = np.meshgrid(x, y)
pos = np.dstack((X_grid, Y_grid))

rv = multivariate_normal(mean, cov)
Z = rv.pdf(pos)

# Dibujar curvas de nivel
axes[0, 0].contour(X_grid, Y_grid, Z, levels=10, cmap='Oranges')

axes[0, 0].set_title(r"$p({\bf z})$")
axes[0, 0].set_xlabel(r"$x_1$")
axes[0, 0].set_ylabel(r"$x_2$")

# =========================
# ======== p(x) ===========
# =========================
axes[0, 1].scatter(X_moons[:, 0], X_moons[:, 1], color='C0')
axes[0, 1].set_title(r"$p({\bf x})$")
axes[0, 1].set_xlabel(r"$x_1$")
axes[0, 1].set_ylabel(r"$x_2$")

# ================================================
# == Sample from p(x,z) through ancestral sampling
# z ~ p(z)
# x ~ p(x|z)
# ================================================
torch.manual_seed(10)
n_samples = 500
vae.eval()
with torch.no_grad():
    z, x_z = vae.sample_from_prior(n_samples)

    # come back to original scale.
    x_z = scaler.inverse_transform(x_z.numpy())


axes[1, 0].plot(z[:, 0], z[:, 1],'x', color='C1', alpha = 0.1)        
axes[1, 0].contour(X_grid, Y_grid, Z, levels=10, cmap='Oranges')
axes[1, 0].set_title(r"Sample from $p({\bf z})$")
axes[1, 0].set_xlabel(r"$z_1$")
axes[1, 0].set_ylabel(r"$z_2$")


axes[1, 1].plot(x_z[:, 0], x_z[:, 1],'x', color='C0', alpha = 0.1)
axes[1, 1].set_title(r"Sample from $p({\bf x}|{\bf z})$")
axes[1, 1].set_xlabel(r"$x_1$")
axes[1, 1].set_ylabel(r"$x_2$")

for s in range(10):
    plot_z, = axes[1, 0].plot(z[s, 0], z[s, 1], 'o', color='C1', markersize = 10)    
    
    grab_frame()
    
    plot_x_z, = axes[1, 1].plot(x_z[s, 0], x_z[s, 1],'o', color='C0', markersize = 10)

    grab_frame()

    plot_z.remove()
    plot_x_z.remove()

writer.close()
plt.close()


In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

### Generating Samples from the model using approximate MCMC

In [ ]:
# Fijar semilla para reproducibilidad
np.random.seed(42)

# Crear figura con dos filas y dos columnas
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=25, codec="libx264")

def grab_frame():
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

# =========================
# == Prior p(z) = N(0,I) ==
# =========================

# Generar muestras 2D de una normal estándar
mean = [0, 0]
cov = [[1, 0], [0, 1]]
samples = np.random.multivariate_normal(mean, cov, 500)

# Scatter de las muestras
axes[0, 0].plot(samples[:, 0], samples[:, 1],'x', color='C1', alpha = 0.1)

# Crear grid para curvas de nivel
x = np.linspace(-4, 4, 200)
y = np.linspace(-4, 4, 200)
X_grid, Y_grid = np.meshgrid(x, y)
pos = np.dstack((X_grid, Y_grid))

rv = multivariate_normal(mean, cov)
Z = rv.pdf(pos)

# Dibujar curvas de nivel
axes[0, 0].contour(X_grid, Y_grid, Z, levels=10, cmap='Oranges')

axes[0, 0].set_title(r"$p({\bf z})$")
axes[0, 0].set_xlabel(r"$z_1$")
axes[0, 0].set_ylabel(r"$z_2$")

# =========================
# ======== p(x) ===========
# =========================
axes[0, 1].plot(X_moons[:, 0], X_moons[:, 1],'x', color='C0', alpha = 0.1)
axes[0, 1].set_title(r"$p({\bf x})$")
axes[0, 1].set_xlabel(r"$x_1$")
axes[0, 1].set_ylabel(r"$x_2$")

# ================================================
# == Sample from p(x,z) through ancestral sampling
# z ~ p(z)
# x ~ p(x|z)
# ================================================
torch.manual_seed(1)
num_steps = 20
n_chains = 3
vae.eval()
with torch.no_grad():
    for x, in sampling_loader:
        # add the point(10,30), normalized, which looks nice
        x_n = torch.tensor(scaler.transform(np.array([[10.,30.]])), dtype = torch.float)
        x = torch.cat([x_n, x], dim=0)
        
        z_x, x_z = vae.run_mcmc(
            x = x, 
            num_steps = num_steps,
            n_chains = n_chains,
            return_mean = True
        )
    # print(z_x.shape) #  [batch_size, n_chains, num_steps, dim] 
    # print(x_z.shape)

## Estimate a Kernel density estimator on q(z|x) to grab idea on what is the posterior
#  looking. Get final half of the chain.
kde = gaussian_kde(z_x[:,:,int(num_steps/2):,:].contiguous().view(-1,data_dim).numpy().T)

# come back to original scale.
x_z_flat = x_z.view(-1, x_z.shape[-1])
x_z_orig = scaler.inverse_transform(x_z_flat)
x_z = torch.tensor(x_z_orig).view(x_z.shape)

## Plot kde level curves
Z = kde(pos.reshape(-1, 2).T).reshape(X_grid.shape)
axes[1,0].contour(X_grid, Y_grid, Z, levels=20,  cmap='Oranges')

## plot all. Pick half of the chain
z_x_plot = z_x[:,:,int(num_steps/2):,:].contiguous().view(-1,data_dim).numpy()
x_z_plot = x_z[:,:,int(num_steps/2):,:].contiguous().view(-1,data_dim).numpy()

axes[1, 0].plot(z_x_plot[:, 0], z_x_plot[:, 1],'x', color='C1', alpha = 0.2)  
axes[1, 0].set_xlim([-4,4])
axes[1, 0].set_ylim([-4,4])
axes[1, 0].set_title(r"Sample from $q({\bf z} \mid {\bf x})$")
axes[1, 0].set_xlabel(r"$z_1$")
axes[1, 0].set_ylabel(r"$z_2$")

axes[1, 1].plot(x_z_plot[:, 0], x_z_plot[:, 1],'x', color='C0', alpha = 0.2)
axes[1, 1].set_title(r"Sample from $p({\bf x}|{\bf z})$")
axes[1, 1].set_xlabel(r"$x_1$")
axes[1, 1].set_ylabel(r"$x_2$")


# =========================
# Animate MCMC trajectories 
# =========================
batch_size, n_chains, num_steps, data_dim = x_z.shape

# Initialize trajectory lists and plot handles
plot_handles = [[None for _ in range(n_chains)] for _ in range(batch_size)]

# Loop over MCMC steps
for b in range(batch_size):
    for c in range(n_chains):
        traj_x_x = []
        traj_x_y = []
        traj_z_x = []
        traj_z_y = []
        plot_handle_x = None

        # initial samples where the chain is started
        init_x_x = x_z[b, c, 0, 0].item() # traj_x_x.append()
        init_x_y = x_z[b, c, 0, 1].item() #traj_x_y.append()

        # highlight the initial samples of the chain
        init_chain, = axes[0,1].plot(init_x_x, init_x_y, 'x', color = 'black', markersize=8)

        traj_x_x.append(init_x_x)
        traj_x_y.append(init_x_y)
        
        for t in range(num_steps-1):
            # append current point to trajectory
            traj_x_x.append(x_z[b, c, t+1, 0].item())
            traj_x_y.append(x_z[b, c, t+1, 1].item())

            traj_z_x.append(z_x[b, c, t, 0].item())
            traj_z_y.append(z_x[b, c, t, 1].item())

            if plot_handle_x is not None:
                plot_handle_z.remove()
            
            # plot trajectory for this chain of this batch element
            plot_handle_z, = axes[1,0].plot(traj_z_x, traj_z_y,
                                          '--x', markersize=4, color='black', alpha=0.5, zorder = 10)
            grab_frame()

            if plot_handle_x is not None:
                plot_handle_x.remove()
            
            plot_handle_x, = axes[1,1].plot(traj_x_x, traj_x_y,
                                          '--x', markersize=4, color='black', alpha=0.5, zorder = 10)

            grab_frame()  # pause to animate each step

        # remove previous line
        plot_handle_x.remove()
        plot_handle_z.remove()
        init_chain.remove()
        
        # highlight final element in the chain
        axes[1, 1].plot(traj_x_x[-1], traj_x_y[-1],'o', markersize = 8, color='C0')
        axes[1, 0].plot(traj_z_x[-1], traj_z_y[-1],'o', markersize = 8, color='C1')

writer.close()
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

## TODO

- Add the theory for the VAE.